## Data Preparation


In [4]:
import pandas as pd


X_train = pd.read_csv("X_train.csv")
y_train = pd.read_csv("y_train.csv")

X_test = pd.read_csv("X_test.csv")
y_test = pd.read_csv("y_test.csv")

In [14]:
print("X_train ticket IDs unique:", X_train["id"].is_unique)
print("y_train label IDs unique:", y_train["id"].is_unique)

print("X_test ticket IDs unique:", X_test["id"].is_unique)
print("y_test label IDs unique:", y_test["id"].is_unique)

# print("Training IDs match:",set(X_train["id"]) == set(y_train["id"]))

# print("Testing IDs match:",set(X_test["id"]) == set(y_test["id"]))

X_train ticket IDs unique: True
y_train label IDs unique: True
X_test ticket IDs unique: True
y_test label IDs unique: True


### Merging the columns of X_train + y_train and X_test + y_test

In [31]:
train_df = X_train.merge(
    y_train,
    on="id",
    how="inner",
    validate="one_to_one"
)

test_df = X_test.merge(
    y_test,
    on="id",
    how="inner",
    validate="one_to_one"
)

print(train_df.shape)
print(test_df.shape)

display(train_df.head())
display(test_df.head())

(1572, 3)
(657, 3)


,id,text,category_truth
0,1919,File Share Access - [TICKET ID] - [NAME]. [NAM...,Fileservice
1,1584,File Share Access - [TICKET ID] - [NAME] ([COM...,Fileservice
2,584,[TICKET ID] - A Support Ticket was forwarded t...,Support general
3,1393,[TICKET ID] - A Support Ticket was forwarded t...,Support general
4,8636,LVS not respondingLVS will not except my passw...,Fileservice


,id,text,category_truth
0,830,[TICKET ID] - New Support Ticket received - S...,Support general
1,1030,[TICKET ID] - New Support Ticket received - R...,Fileservice
2,1184,[TICKET ID] - New Support Ticket received - U...,Software
3,5570,The following system must be set to End Of Lif...,EOL
4,4585,File Share Access - [TICKET ID] - [NAME] ([COM...,Fileservice


### Ensuring that no NULL values exist and count of categories

In [37]:
print("Training data missing values:")
print(train_df[["text", "category_truth"]].isnull().sum())

print("\nTesting data missing values:")
print(test_df[["text", "category_truth"]].isnull().sum())

print("\nTraining category distribution:")
print(train_df["category_truth"].value_counts())

print("\nTesting category distribution:")
print(test_df["category_truth"].value_counts())

Training data missing values:
text              0
category_truth    0
dtype: int64

Testing data missing values:
text              0
category_truth    0
dtype: int64

Training category distribution:
category_truth
Fileservice          546
Support general      541
Software             164
O365                 127
Active Directory      84
Computer-Services     65
EOL                   45
Name: count, dtype: int64

Testing category distribution:
category_truth
Support general      221
Fileservice          138
O365                  92
Software              58
EOL                   58
Active Directory      49
Computer-Services     41
Name: count, dtype: int64


In [43]:
X_train_text = train_df["text"].astype(str)
y_train = train_df["category_truth"]

X_test_text = test_df["text"].astype(str)
y_test = test_df["category_truth"]

print("Training tickets:", len(X_train_text))
print("Testing tickets:", len(X_test_text))
print("Categories:", y_train.unique())

Training tickets: 1572
Testing tickets: 657
Categories: ['Fileservice' 'Support general' 'Software' 'EOL' 'O365'
 'Active Directory' 'Computer-Services']


## Model developing


In [63]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        max_features=20000
    )),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

baseline_model.fit(X_train_text, y_train)

# print("Baseline model trained successfully!")

,steps,"[('tfidf', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [72]:
from sklearn.metrics import accuracy_score,classification_report,f1_score

y_pred = baseline_model.predict(X_test_text)

print("Test accuracy:", round(accuracy_score(y_test, y_pred), 4))

print("Macro F1-score:", round(f1_score(y_test, y_pred, average="macro"), 4))

print("\nClassification report:\n")

print(classification_report(y_test,y_pred,zero_division=0))

Test accuracy: 0.7428
Macro F1-score: 0.657

Classification report:

                   precision    recall  f1-score   support

 Active Directory       1.00      0.20      0.34        49
Computer-Services       0.89      0.41      0.57        41
              EOL       1.00      1.00      1.00        58
      Fileservice       1.00      0.91      0.95       138
             O365       0.79      0.52      0.63        92
         Software       0.57      0.28      0.37        58
  Support general       0.60      0.96      0.74       221

         accuracy                           0.74       657
        macro avg       0.84      0.61      0.66       657
     weighted avg       0.79      0.74      0.72       657



In [82]:
results = test_df[["text", "category_truth"]].copy()

results["predicted_category"] = y_pred
results["correct"] = (results["category_truth"] ==results["predicted_category"])

display(results.head(10))

,text,category_truth,predicted_category,correct
0,[TICKET ID] - New Support Ticket received - S...,Support general,Support general,True
1,[TICKET ID] - New Support Ticket received - R...,Fileservice,Support general,False
2,[TICKET ID] - New Support Ticket received - U...,Software,Support general,False
3,The following system must be set to End Of Lif...,EOL,EOL,True
4,File Share Access - [TICKET ID] - [NAME] ([COM...,Fileservice,Fileservice,True
5,The following system must be set to End Of Lif...,EOL,EOL,True
6,[TICKET ID] - A Support Ticket was forwarded t...,Software,Support general,False
7,[TICKET ID] - A Support Ticket was forwarded t...,Support general,Support general,True
8,[TICKET ID] - New Support Ticket received - L...,Support general,Support general,True
9,[TICKET ID] - New Support Ticket received - F...,Fileservice,Fileservice,True


In [88]:
from sklearn.metrics import confusion_matrix

labels = sorted(y_test.unique())

cm = confusion_matrix(
    y_test,
    y_pred,
    labels=labels
)

cm_df = pd.DataFrame(
    cm,
    index=[f"Actual: {label}" for label in labels],
    columns=[f"Predicted: {label}" for label in labels]
)

display(cm_df)

,Predicted: Active Directory,Predicted: Computer-Services,Predicted: EOL,Predicted: Fileservice,Predicted: O365,Predicted: Software,Predicted: Support general
Actual: Active Directory,10,0,0,0,2,1,36
Actual: Computer-Services,0,17,0,0,2,4,18
Actual: EOL,0,0,58,0,0,0,0
Actual: Fileservice,0,0,0,126,2,2,8
Actual: O365,0,0,0,0,48,0,44
Actual: Software,0,1,0,0,5,16,36
Actual: Support general,0,1,0,0,2,5,213


In [93]:
errors = results[results["correct"] == False].copy()

print("Incorrect predictions:", len(errors))
print("\nMost common incorrect predictions:")

display(errors.groupby(["category_truth", "predicted_category"]).size().sort_values(ascending=False).head(15).rename("count").reset_index())

Incorrect predictions: 169

Most common incorrect predictions:


,category_truth,predicted_category,count
0,O365,Support general,44
1,Active Directory,Support general,36
2,Software,Support general,36
3,Computer-Services,Support general,18
4,Fileservice,Support general,8
5,Software,O365,5
6,Support general,Software,5
7,Computer-Services,Software,4
8,Active Directory,O365,2
9,Fileservice,Software,2


In [94]:
pd.set_option("display.max_colwidth", 250)

display(errors[["text", "category_truth", "predicted_category"]].head(10))

,text,category_truth,predicted_category
1,[TICKET ID] - New Support Ticket received - R Drive Permission,Fileservice,Support general
2,[TICKET ID] - New Support Ticket received - Update Rstudio,Software,Support general
6,[TICKET ID] - A Support Ticket was forwarded to your role - Kein Zugriff mehr auf [COMPANY] Intranet. (OKTA-APP nicht zugewiesen),Software,Support general
10,[TICKET ID] - A Support Ticket was forwarded to your role - Instalação Impressora,Computer-Services,O365
11,[TICKET ID] - New Support Ticket received - New account,Active Directory,Support general
22,reserva DHCPPoderia fazer uma reserva no DHCP? [LOCATION][MAC_ADDRESS] é uma impressora.,Support general,Computer-Services
24,[TICKET ID] - New Support Ticket received - Okta Verify Reset,Software,Support general
30,[TICKET ID] - New Support Ticket received - no accede a el mail,O365,Support general
35,[TICKET ID] - New Support Ticket received - Cadastrar usuário + compra de equipamento -[LOCATION]13494 - [NAME],Active Directory,Support general
37,[TICKET ID] - Configure Windows account and additional services for [NAME] ([COMPANY]; [LOCATION]) (Order Request ID [TICKET ID]),Active Directory,Support general


In [102]:
display(cm_df)

print("Total test tickets:", cm.sum())
print("Correct predictions:", cm.trace())
print("Incorrect predictions:", cm.sum() - cm.trace())

,Active Directory,Computer-Services,EOL,Fileservice,O365,Software,Support general
Active Directory,10,0,0,0,2,1,36
Computer-Services,0,17,0,0,2,4,18
EOL,0,0,58,0,0,0,0
Fileservice,0,0,0,126,2,2,8
O365,0,0,0,0,48,0,44
Software,0,1,0,0,5,16,36
Support general,0,1,0,0,2,5,213


Total test tickets: 657
Correct predictions: 488
Incorrect predictions: 169
